In [3]:
import numpy as np
import pandas as pd 
import emoji

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical


In [4]:
data = pd.read_csv("./data/emoji_data.csv", header = None)

In [5]:
data.head()
data

,0,1
0,French macaroon is so tasty,4
1,work is horrible,3
2,I am upset,3
3,throw the ball,1
4,Good joke,2
...,...,...
178,lets brunch some day,4
179,dance with me,2
180,she is a bully,3
181,she plays baseball,1


In [22]:
emoji_dict = {0: ":red_heart:", 
              1: ":baseball:", 
              2: ":grinning_face_with_big_eyes:",
              3: ":disappointed_face:",
              4: ":fork_and_knife_with_plate:"
             }
def label_to_emoji(label):
    return emoji.emojize(emoji_dict[label])

In [23]:
X = data[0].values
Y = data[1].values

In [24]:
file = open('./data/glove.6B.100d.txt', 'r', encoding='utf8')
content = file.readlines()
file.close()

In [25]:
embeddings = {}

for line in content:
    line = line.split()
    embeddings[line[0]] = np.array(line[1:], dtype = float)

In [26]:
def get_maxlen(data):
    maxlen = 0
    for sent in data:
        maxlen = max(maxlen, len(sent))
    return maxlen

In [27]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X)
word2index = tokenizer.word_index
word2index

{'i': 1,
 'you': 2,
 'is': 3,
 'the': 4,
 'a': 5,
 'so': 6,
 'am': 7,
 'my': 8,
 'to': 9,
 'this': 10,
 'are': 11,
 'ha': 12,
 'for': 13,
 'she': 14,
 'he': 15,
 'me': 16,
 'not': 17,
 'love': 18,
 'your': 19,
 'want': 20,
 'have': 21,
 'it': 22,
 'got': 23,
 'like': 24,
 'did': 25,
 'baseball': 26,
 'food': 27,
 'was': 28,
 'do': 29,
 'joke': 30,
 'stop': 31,
 'will': 32,
 'miss': 33,
 'life': 34,
 'ball': 35,
 'good': 36,
 'what': 37,
 'go': 38,
 'job': 39,
 'funny': 40,
 'bad': 41,
 'day': 42,
 'great': 43,
 'dinner': 44,
 'that': 45,
 'with': 46,
 'at': 47,
 'of': 48,
 'game': 49,
 'we': 50,
 'again': 51,
 'said': 52,
 'yes': 53,
 'lol': 54,
 'and': 55,
 'down': 56,
 'had': 57,
 'her': 58,
 'fun': 59,
 'smile': 60,
 'lot': 61,
 'working': 62,
 'him': 63,
 'cute': 64,
 'on': 65,
 'lets': 66,
 'messing': 67,
 'us': 68,
 'play': 69,
 'exercise': 70,
 'lost': 71,
 'never': 72,
 'where': 73,
 'can': 74,
 'well': 75,
 'much': 76,
 'valentine': 77,
 'restaurant': 78,
 'awesome': 79,
 'lik

In [28]:
Xtokens = tokenizer.texts_to_sequences(X)
maxlen = get_maxlen(Xtokens)
Xtrain = pad_sequences(Xtokens, maxlen = maxlen, padding = 'post')

In [29]:
Ytrain = to_categorical(Y)

In [30]:
embed_size = 100 
embedding_matrix = np.zeros((len(word2index) + 1, embed_size))

for word, i in word2index.items():
    embed_vector = embeddings.get(word)
    if embed_vector is not None:
        embedding_matrix[i] = embed_vector

In [31]:
model = Sequential([
    Embedding(input_dim=len(word2index) + 1,
              output_dim=embed_size,
              input_length=maxlen,   
              weights=[embedding_matrix],
              trainable=False),
    
    LSTM(units=16, return_sequences=True),
    LSTM(units=4),
    Dense(5, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=["accuracy"])

/home/berk/miniconda3/envs/yapayzeka/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [32]:
model.fit(Xtrain, Ytrain, epochs=100)

Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2186 - loss: 1.6060
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.2842 - loss: 1.5844
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.2896 - loss: 1.5719
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.2896 - loss: 1.5603
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3005 - loss: 1.5488
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3224 - loss: 1.5334
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3989 - loss: 1.5157
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4153 - loss: 1.4885
Epoch 9/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4208 - loss: 1.4603
Epoch 10/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4208 - loss: 1.4316
Epoch 11/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4044 - loss: 1.4003
Epoch 12/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4372 - lo

In [34]:
test =["I feel good", "I feel very bad", "Mehmet is ugly", "I am starving", "I feel ill", "I love footbal", "lets eat dinner"]
test_seq = tokenizer.texts_to_sequences(test)
Xtest = pad_sequences(test_seq, maxlen = maxlen, padding = "post", truncating = "post")

y_pred = model.predict(Xtest)
y_pred = np.argmax(y_pred, axis = 1)

for i in range(len(test)):
    print(test[i], label_to_emoji(y_pred[i]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
I feel good 😃
I feel very bad 😞
Mehmet is ugly 😞
I am starving 🍽️
I feel ill 😃
I love footbal ❤️
lets eat dinner 🍽️
